# End-to-End Order Fulfillment Time Prediction for Costco

This project predicts order fulfillment times for Costco's online orders and identifies bottlenecks in the fulfillment process. We'll build a Streamlit application that allows warehouse managers to input order details and receive time estimates.

## Project Architecture

1. **Data Collection**: Synthetic dataset simulating Costco's order fulfillment pipeline
2. **Feature Engineering**: Create meaningful predictors from raw order data
3. **Model Training**: Regression models to predict fulfillment time
4. **Bottleneck Analysis**: Identify process stages causing delays
5. **Streamlit App**: Interactive dashboard for predictions and insights

## Dataset Creation

Let's create a synthetic dataset that mimics Costco's order fulfillment process with these components:

```python
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Configuration
NUM_ORDERS = 10000
START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2023, 12, 31)

# Generate synthetic products
products = pd.DataFrame({
    'product_id': range(1, 501),
    'category': np.random.choice([
        'Electronics', 'Grocery', 'Home', 'Appliances', 
        'Furniture', 'Clothing', 'Seasonal', 'Kirkland Signature'
    ], size=500),
    'weight_lbs': np.round(np.random.uniform(0.5, 50, size=500),  # Weight in pounds
    'is_bulk': np.random.choice([0, 1], size=500, p=[0.3, 0.7]),  # 70% bulk items
    'is_perishable': np.random.choice([0, 1], size=500, p=[0.8, 0.2])  # 20% perishable
})

# Generate synthetic warehouses
warehouses = pd.DataFrame({
    'warehouse_id': range(1, 11),
    'region': np.random.choice(['West', 'Midwest', 'South', 'Northeast'], size=10),
    'capacity': np.random.randint(50000, 200000, size=10),  # Square footage
    'staff_level': np.random.choice(['Low', 'Medium', 'High'], size=10)
})

# Generate synthetic orders
def generate_orders(num_orders):
    orders = []
    for i in range(1, num_orders + 1):
        order_date = START_DATE + timedelta(days=random.randint(0, (END_DATE - START_DATE).days))
        warehouse = random.choice(warehouses['warehouse_id'].values)
        membership_type = random.choice(['Gold Star', 'Executive', 'Business'])
        is_weekend = 1 if order_date.weekday() >= 5 else 0
        is_holiday = 1 if order_date.month == 12 and order_date.day in range(15, 26) else 0  # Holiday season
        
        # Generate random fulfillment stages with timestamps
        processing_time = random.randint(1, 48)  # Hours
        picking_time = random.randint(1, 24)
        packing_time = random.randint(1, 12)
        shipping_time = random.randint(1, 72)
        
        total_time = processing_time + picking_time + packing_time + shipping_time
        
        orders.append({
            'order_id': i,
            'order_date': order_date,
            'warehouse_id': warehouse,
            'membership_type': membership_type,
            'num_items': random.randint(1, 20),
            'total_weight': random.uniform(5, 200),
            'is_weekend': is_weekend,
            'is_holiday': is_holiday,
            'processing_time_hrs': processing_time,
            'picking_time_hrs': picking_time,
            'packing_time_hrs': packing_time,
            'shipping_time_hrs': shipping_time,
            'total_fulfillment_time_hrs': total_time,
            'on_time': 1 if total_time <= 48 else 0  # 48hrs SLA
        })
    return pd.DataFrame(orders)

orders_df = generate_orders(NUM_ORDERS)

# Save datasets
products.to_csv('costco_products.csv', index=False)
warehouses.to_csv('costco_warehouses.csv', index=False)
orders_df.to_csv('costco_orders.csv', index=False)
```

## Data Preprocessing & Feature Engineering

```python
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# Load data
orders = pd.read_csv('costco_orders.csv')
warehouses = pd.read_csv('costco_warehouses.csv')

# Merge with warehouse data
orders = orders.merge(warehouses, on='warehouse_id')

# Feature engineering
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders['order_month'] = orders['order_date'].dt.month
orders['order_dayofweek'] = orders['order_date'].dt.dayofweek
orders['order_hour'] = orders['order_date'].dt.hour

# Prepare features and target
X = orders.drop(['order_id', 'order_date', 'total_fulfillment_time_hrs', 'on_time'], axis=1)
y = orders['total_fulfillment_time_hrs']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing pipeline
numeric_features = ['num_items', 'total_weight', 'is_weekend', 'is_holiday', 'order_month', 'order_dayofweek', 'order_hour']
categorical_features = ['warehouse_id', 'membership_type', 'region', 'staff_level']

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Model pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train model
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f} hours")
print(f"R2 Score: {r2_score(y_test, y_pred):.2f}")

# Save model
joblib.dump(model, 'costco_fulfillment_model.pkl')
```

## Streamlit Application

```python
import streamlit as st
import pandas as pd
import joblib
from datetime import datetime
import matplotlib.pyplot as plt

# Load model and data
model = joblib.load('costco_fulfillment_model.pkl')
warehouses = pd.read_csv('costco_warehouses.csv')

# App title
st.title("Costco Order Fulfillment Time Predictor")
st.markdown("""
This tool predicts order fulfillment times and identifies bottlenecks in Costco's supply chain.
""")

# Sidebar for input parameters
st.sidebar.header("Order Details")

# Input form
with st.form("order_form"):
    col1, col2 = st.columns(2)
    
    with col1:
        warehouse_id = st.selectbox(
            "Warehouse ID",
            options=warehouses['warehouse_id'].unique()
        )
        membership_type = st.selectbox(
            "Membership Type",
            options=['Gold Star', 'Executive', 'Business']
        )
        num_items = st.number_input(
            "Number of Items",
            min_value=1,
            max_value=50,
            value=5
        )
        
    with col2:
        total_weight = st.number_input(
            "Total Weight (lbs)",
            min_value=1.0,
            max_value=500.0,
            value=25.0
        )
        order_date = st.date_input(
            "Order Date",
            value=datetime.now()
        )
        order_time = st.time_input(
            "Order Time",
            value=datetime.now().time()
        )
    
    submit_button = st.form_submit_button("Predict Fulfillment Time")

# Process input and make prediction
if submit_button:
    # Prepare input data
    order_datetime = datetime.combine(order_date, order_time)
    is_weekend = 1 if order_datetime.weekday() >= 5 else 0
    is_holiday = 1 if order_datetime.month == 12 and order_datetime.day in range(15, 26) else 0
    
    warehouse_data = warehouses[warehouses['warehouse_id'] == warehouse_id].iloc[0]
    
    input_data = pd.DataFrame([{
        'warehouse_id': warehouse_id,
        'membership_type': membership_type,
        'num_items': num_items,
        'total_weight': total_weight,
        'is_weekend': is_weekend,
        'is_holiday': is_holiday,
        'order_month': order_datetime.month,
        'order_dayofweek': order_datetime.weekday(),
        'order_hour': order_datetime.hour,
        'region': warehouse_data['region'],
        'staff_level': warehouse_data['staff_level']
    }])
    
    # Make prediction
    prediction = model.predict(input_data)[0]
    
    # Display results
    st.subheader("Prediction Results")
    col1, col2 = st.columns(2)
    
    with col1:
        st.metric("Predicted Fulfillment Time", f"{prediction:.1f} hours")
        
        if prediction <= 48:
            st.success("This order is predicted to meet the 48-hour SLA")
        else:
            st.error("This order is at risk of missing the 48-hour SLA")
    
    with col2:
        # Get feature importances
        feature_importances = model.named_steps['regressor'].feature_importances_
        feature_names = (model.named_steps['preprocessor']
                        .named_transformers_['cat']
                        .named_steps['onehot']
                        .get_feature_names_out(categorical_features))
        
        all_feature_names = numeric_features + list(feature_names)
        importance_df = pd.DataFrame({
            'Feature': all_feature_names,
            'Importance': feature_importances
        }).sort_values('Importance', ascending=False).head(10)
        
        fig, ax = plt.subplots()
        ax.barh(importance_df['Feature'], importance_df['Importance'])
        ax.set_xlabel('Importance')
        ax.set_title('Top Factors Affecting Fulfillment Time')
        st.pyplot(fig)
    
    # Bottleneck analysis
    st.subheader("Bottleneck Analysis")
    
    # Simulate stage times (in a real app, these would come from model components)
    stages = {
        'Processing': prediction * 0.3,
        'Picking': prediction * 0.4,
        'Packing': prediction * 0.1,
        'Shipping': prediction * 0.2
    }
    
    bottleneck = max(stages, key=stages.get)
    
    col1, col2 = st.columns(2)
    
    with col1:
        st.write("Time by Stage:")
        for stage, time in stages.items():
            st.write(f"- {stage}: {time:.1f} hours")
    
    with col2:
        st.warning(f"Potential Bottleneck: {bottleneck} stage")
        
        if bottleneck == 'Picking':
            st.write("Recommendations:")
            st.write("- Optimize warehouse layout")
            st.write("- Increase picker staff during peak hours")
            st.write("- Implement batch picking for multi-item orders")
        elif bottleneck == 'Shipping':
            st.write("Recommendations:")
            st.write("- Negotiate better carrier contracts")
            st.write("- Implement regional fulfillment centers")
            st.write("- Offer tiered shipping options")

# Add historical data visualization
st.subheader("Historical Performance")
show_historical = st.checkbox("Show historical fulfillment times")

if show_historical:
    orders = pd.read_csv('costco_orders.csv')
    orders['order_date'] = pd.to_datetime(orders['order_date'])
    
    time_period = st.selectbox(
        "Time Period",
        options=['Last 7 days', 'Last 30 days', 'Last 90 days', 'Last year']
    )
    
    if time_period == 'Last 7 days':
        cutoff = datetime.now() - timedelta(days=7)
    elif time_period == 'Last 30 days':
        cutoff = datetime.now() - timedelta(days=30)
    elif time_period == 'Last 90 days':
        cutoff = datetime.now() - timedelta(days=90)
    else:
        cutoff = datetime.now() - timedelta(days=365)
    
    filtered = orders[orders['order_date'] >= cutoff]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(filtered['order_date'], filtered['total_fulfillment_time_hrs'], alpha=0.5)
    ax.axhline(y=48, color='r', linestyle='--', label='48-hour SLA')
    ax.set_xlabel('Order Date')
    ax.set_ylabel('Fulfillment Time (hours)')
    ax.set_title('Actual Fulfillment Times')
    ax.legend()
    st.pyplot(fig)
```

## How to Run the Application

1. Save the dataset generation code to `generate_data.py`
2. Save the modeling code to `train_model.py`
3. Save the Streamlit app code to `app.py`
4. Install required packages:
```bash
pip install pandas numpy scikit-learn streamlit matplotlib joblib
```
5. Generate the data:
```bash
python generate_data.py
```
6. Train the model:
```bash
python train_model.py
```
7. Run the Streamlit app:
```bash
streamlit run app.py
```

## Key Features of the Application

1. **Interactive Prediction**: Input order details and get real-time fulfillment time predictions
2. **Bottleneck Identification**: Highlights which stage is causing delays
3. **Recommendation Engine**: Provides actionable insights to improve fulfillment
4. **Historical Analysis**: Visualizes past performance against SLA
5. **Feature Importance**: Shows which factors most impact fulfillment times

This end-to-end solution provides Costco warehouse managers with actionable insights to optimize their order fulfillment process, reduce delays, and improve customer satisfaction.

Dataset Creation

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Configuration
NUM_ORDERS = 10000
START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2023, 12, 31)

# Generate synthetic products
products = pd.DataFrame({
    'product_id': range(1, 501),
    'category': np.random.choice([
        'Electronics', 'Grocery', 'Home', 'Appliances', 
        'Furniture', 'Clothing', 'Seasonal', 'Kirkland Signature'
    ], size=500),
    'weight_lbs': np.round(np.random.uniform(0.5, 50, size=500), 2),  # Weight in pounds
    'is_bulk': np.random.choice([0, 1], size=500, p=[0.3, 0.7]),  # 70% bulk items
    'is_perishable': np.random.choice([0, 1], size=500, p=[0.8, 0.2])  # 20% perishable
})

# Generate synthetic warehouses
warehouses = pd.DataFrame({
    'warehouse_id': range(1, 11),
    'region': np.random.choice(['West', 'Midwest', 'South', 'Northeast'], size=10),
    'capacity': np.random.randint(50000, 200000, size=10),  # Square footage
    'staff_level': np.random.choice(['Low', 'Medium', 'High'], size=10)
})

# Generate synthetic orders
def generate_orders(num_orders):
    orders = []
    for i in range(1, num_orders + 1):
        order_date = START_DATE + timedelta(days=random.randint(0, (END_DATE - START_DATE).days))
        warehouse = random.choice(warehouses['warehouse_id'].values)
        membership_type = random.choice(['Gold Star', 'Executive', 'Business'])
        is_weekend = 1 if order_date.weekday() >= 5 else 0
        is_holiday = 1 if order_date.month == 12 and order_date.day in range(15, 26) else 0

        # 💡 Link to product
        product = products.sample(1).iloc[0]
        product_id = product['product_id']
        product_weight = product['weight_lbs']
        num_items = random.randint(1, 20)
        total_weight = round(product_weight * num_items, 2)

        # Fulfillment times
        processing_time = random.randint(1, 48)
        picking_time = random.randint(1, 24)
        packing_time = random.randint(1, 12)
        shipping_time = random.randint(1, 72)
        total_time = processing_time + picking_time + packing_time + shipping_time

        orders.append({
            'order_id': i,
            'order_date': order_date,
            'warehouse_id': warehouse,
            'membership_type': membership_type,
            'product_id': product_id,               # ✅ Added product_id
            'category': product['category'],        # Optional: add category info
            'num_items': num_items,
            'total_weight': total_weight,
            'is_weekend': is_weekend,
            'is_holiday': is_holiday,
            'processing_time_hrs': processing_time,
            'picking_time_hrs': picking_time,
            'packing_time_hrs': packing_time,
            'shipping_time_hrs': shipping_time,
            'total_fulfillment_time_hrs': total_time,
            'on_time': 1 if total_time <= 48 else 0
        })
    return pd.DataFrame(orders)


orders_df = generate_orders(NUM_ORDERS)

# Save datasets
products.to_csv('costco_products.csv', index=False)
warehouses.to_csv('costco_warehouses.csv', index=False)
orders_df.to_csv('costco_orders.csv', index=False)


Data Preprocessing & Feature Engineering

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# Load data
orders = pd.read_csv('costco_orders.csv')
warehouses = pd.read_csv('costco_warehouses.csv')
products = pd.read_csv('costco_products.csv')  # 👈 Ensure this file includes product_id, category, is_bulk, etc.

# Merge with warehouse and product data
orders = orders.merge(warehouses, on='warehouse_id')
orders = orders.merge(products, on='product_id')  # 👈 Merge product info

# Feature engineering
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders['order_month'] = orders['order_date'].dt.month
orders['order_dayofweek'] = orders['order_date'].dt.dayofweek
orders['order_hour'] = orders['order_date'].dt.hour

# Prepare features and target
X = orders.drop(['order_id', 'order_date', 'total_fulfillment_time_hrs', 'on_time'], axis=1)
y = orders['total_fulfillment_time_hrs']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing pipeline
numeric_features = [
    'num_items', 'total_weight', 'is_weekend', 'is_holiday',
    'order_month', 'order_dayofweek', 'order_hour'
]

# Optionally include product-based numeric features
if 'weight_lbs' in X.columns:
    numeric_features.append('weight_lbs')

categorical_features = [
    'warehouse_id', 'membership_type', 'region', 'staff_level',
    'product_id'  # 👈 include product_id as categorical
]

# Optional: include category or other product features
if 'category' in X.columns:
    categorical_features.append('category')
if 'is_bulk' in X.columns:
    numeric_features.append('is_bulk')
if 'is_perishable' in X.columns:
    numeric_features.append('is_perishable')

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Model pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Train model
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f} hours")
print(f"R2 Score: {r2_score(y_test, y_pred):.2f}")

# Save model
joblib.dump(model, 'costco_fulfillment_model_1.pkl')


MAE: 21.03 hours
R2 Score: -0.04


['costco_fulfillment_model_1.pkl']

Streamlit Application

In [4]:
import streamlit as st
import pandas as pd
import joblib
from datetime import datetime
import matplotlib.pyplot as plt

# Load model and data
model = joblib.load('costco_fulfillment_model.pkl')
warehouses = pd.read_csv('costco_warehouses.csv')

# App title
st.title("Costco Order Fulfillment Time Predictor")
st.markdown("""
This tool predicts order fulfillment times and identifies bottlenecks in Costco's supply chain.
""")

# Sidebar for input parameters
st.sidebar.header("Order Details")

# Input form
with st.form("order_form"):
    col1, col2 = st.columns(2)
    
    with col1:
        warehouse_id = st.selectbox(
            "Warehouse ID",
            options=warehouses['warehouse_id'].unique()
        )
        membership_type = st.selectbox(
            "Membership Type",
            options=['Gold Star', 'Executive', 'Business']
        )
        num_items = st.number_input(
            "Number of Items",
            min_value=1,
            max_value=50,
            value=5
        )
        
    with col2:
        total_weight = st.number_input(
            "Total Weight (lbs)",
            min_value=1.0,
            max_value=500.0,
            value=25.0
        )
        order_date = st.date_input(
            "Order Date",
            value=datetime.now()
        )
        order_time = st.time_input(
            "Order Time",
            value=datetime.now().time()
        )
    
    submit_button = st.form_submit_button("Predict Fulfillment Time")

# Process input and make prediction
if submit_button:
    # Prepare input data
    order_datetime = datetime.combine(order_date, order_time)
    is_weekend = 1 if order_datetime.weekday() >= 5 else 0
    is_holiday = 1 if order_datetime.month == 12 and order_datetime.day in range(15, 26) else 0
    
    warehouse_data = warehouses[warehouses['warehouse_id'] == warehouse_id].iloc[0]
    
    input_data = pd.DataFrame([{
        'warehouse_id': warehouse_id,
        'membership_type': membership_type,
        'num_items': num_items,
        'total_weight': total_weight,
        'is_weekend': is_weekend,
        'is_holiday': is_holiday,
        'order_month': order_datetime.month,
        'order_dayofweek': order_datetime.weekday(),
        'order_hour': order_datetime.hour,
        'region': warehouse_data['region'],
        'staff_level': warehouse_data['staff_level']
    }])
    
    # Make prediction
    prediction = model.predict(input_data)[0]
    
    # Display results
    st.subheader("Prediction Results")
    col1, col2 = st.columns(2)
    
    with col1:
        st.metric("Predicted Fulfillment Time", f"{prediction:.1f} hours")
        
        if prediction <= 48:
            st.success("This order is predicted to meet the 48-hour SLA")
        else:
            st.error("This order is at risk of missing the 48-hour SLA")
    
    with col2:
        # Get feature importances
        feature_importances = model.named_steps['regressor'].feature_importances_
        feature_names = (model.named_steps['preprocessor']
                        .named_transformers_['cat']
                        .named_steps['onehot']
                        .get_feature_names_out(categorical_features))
        
        all_feature_names = numeric_features + list(feature_names)
        importance_df = pd.DataFrame({
            'Feature': all_feature_names,
            'Importance': feature_importances
        }).sort_values('Importance', ascending=False).head(10)
        
        fig, ax = plt.subplots()
        ax.barh(importance_df['Feature'], importance_df['Importance'])
        ax.set_xlabel('Importance')
        ax.set_title('Top Factors Affecting Fulfillment Time')
        st.pyplot(fig)
    
    # Bottleneck analysis
    st.subheader("Bottleneck Analysis")
    
    # Simulate stage times (in a real app, these would come from model components)
    stages = {
        'Processing': prediction * 0.3,
        'Picking': prediction * 0.4,
        'Packing': prediction * 0.1,
        'Shipping': prediction * 0.2
    }
    
    bottleneck = max(stages, key=stages.get)
    
    col1, col2 = st.columns(2)
    
    with col1:
        st.write("Time by Stage:")
        for stage, time in stages.items():
            st.write(f"- {stage}: {time:.1f} hours")
    
    with col2:
        st.warning(f"Potential Bottleneck: {bottleneck} stage")
        
        if bottleneck == 'Picking':
            st.write("Recommendations:")
            st.write("- Optimize warehouse layout")
            st.write("- Increase picker staff during peak hours")
            st.write("- Implement batch picking for multi-item orders")
        elif bottleneck == 'Shipping':
            st.write("Recommendations:")
            st.write("- Negotiate better carrier contracts")
            st.write("- Implement regional fulfillment centers")
            st.write("- Offer tiered shipping options")

# Add historical data visualization
st.subheader("Historical Performance")
show_historical = st.checkbox("Show historical fulfillment times")

if show_historical:
    orders = pd.read_csv('costco_orders.csv')
    orders['order_date'] = pd.to_datetime(orders['order_date'])
    
    time_period = st.selectbox(
        "Time Period",
        options=['Last 7 days', 'Last 30 days', 'Last 90 days', 'Last year']
    )
    
    if time_period == 'Last 7 days':
        cutoff = datetime.now() - timedelta(days=7)
    elif time_period == 'Last 30 days':
        cutoff = datetime.now() - timedelta(days=30)
    elif time_period == 'Last 90 days':
        cutoff = datetime.now() - timedelta(days=90)
    else:
        cutoff = datetime.now() - timedelta(days=365)
    
    filtered = orders[orders['order_date'] >= cutoff]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(filtered['order_date'], filtered['total_fulfillment_time_hrs'], alpha=0.5)
    ax.axhline(y=48, color='r', linestyle='--', label='48-hour SLA')
    ax.set_xlabel('Order Date')
    ax.set_ylabel('Fulfillment Time (hours)')
    ax.set_title('Actual Fulfillment Times')
    ax.legend()
    st.pyplot(fig)

2025-07-22 23:11:39.239 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-22 23:11:39.318 
  command:

    streamlit run /Users/gvijaykumarachary/.pyenv/versions/3.10.13/lib/python3.10/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-07-22 23:11:39.318 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-22 23:11:39.318 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-22 23:11:39.319 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-22 23:11:39.319 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-22 23:11:39.319 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-22 2